In [162]:
import torch
import torch.nn as nn
import math

import torch.nn.functional as F

import re
import random
from collections import Counter
from torch.utils.data import Dataset,DataLoader

device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [163]:
pairs = [
    ("hello", "नमस्ते"),
    ("how are you", "तपाईंलाई कस्तो छ"),
    ("i am fine", "मलाई सन्चै छ"),
    ("good morning", "शुभ प्रभात"),
    ("good night", "शुभ रात्री"),
    ("thank you", "धन्यवाद"),
    ("i am learning", "म सिक्दै छु"),
    ("i love machine learning", "मलाई मेसिन लर्निङ मन पर्छ"),
    ("what is your name", "तपाईंको नाम के हो"),
    ("my name is john", "मेरो नाम जोन हो"),
    ("i like python", "मलाई पाइथन मन पर्छ"),
    ("machine learning is interesting", "मेसिन लर्निङ रोचक छ"),
    ("how old are you", "तपाईंको उमेर कति हो"),
    ("where are you going", "तपाईं कहाँ जाँदै हुनुहुन्छ"),
    ("i am a student", "म विद्यार्थी हुँ"),
    ("she is happy", "उनी खुसी छिन्"),
    ("he is a teacher", "उनी शिक्षक हुन्"),
    ("i want to learn", "म सिक्न चाहन्छु"),
    ("see you tomorrow", "भोलि भेटौँला")
]

In [164]:
def tokenize(sentence):
    sentence=sentence.lower()
    sentence=re.sub(r"([?.!,])",
                    r" \1",# adds white space 
                    sentence)
    
    return sentence.strip().split()

In [165]:
sentence="I love machine learning"
print(tokenize(sentence))

['i', 'love', 'machine', 'learning']


In [166]:
SPECIAL_TOKENS=[
    "<PAD>",
    "<UNK>",
    "<BOS",
    "<EOS"
]

In [167]:
def build_vocab(sentences):
    counter=Counter()
    for sentence in sentences:
        counter.update(tokenize(sentence))
        print(counter)
        
    vocab = {
        token: idx
        for idx,token in enumerate(SPECIAL_TOKENS)
    }
    
    for word in counter:
        if word not in vocab:
            vocab[word]=len(vocab)
    return vocab

In [168]:
src_vocab=build_vocab([src for src,tgt in pairs])

Counter({'hello': 1})
Counter({'hello': 1, 'how': 1, 'are': 1, 'you': 1})
Counter({'hello': 1, 'how': 1, 'are': 1, 'you': 1, 'i': 1, 'am': 1, 'fine': 1})
Counter({'hello': 1, 'how': 1, 'are': 1, 'you': 1, 'i': 1, 'am': 1, 'fine': 1, 'good': 1, 'morning': 1})
Counter({'good': 2, 'hello': 1, 'how': 1, 'are': 1, 'you': 1, 'i': 1, 'am': 1, 'fine': 1, 'morning': 1, 'night': 1})
Counter({'you': 2, 'good': 2, 'hello': 1, 'how': 1, 'are': 1, 'i': 1, 'am': 1, 'fine': 1, 'morning': 1, 'night': 1, 'thank': 1})
Counter({'you': 2, 'i': 2, 'am': 2, 'good': 2, 'hello': 1, 'how': 1, 'are': 1, 'fine': 1, 'morning': 1, 'night': 1, 'thank': 1, 'learning': 1})
Counter({'i': 3, 'you': 2, 'am': 2, 'good': 2, 'learning': 2, 'hello': 1, 'how': 1, 'are': 1, 'fine': 1, 'morning': 1, 'night': 1, 'thank': 1, 'love': 1, 'machine': 1})
Counter({'i': 3, 'you': 2, 'am': 2, 'good': 2, 'learning': 2, 'hello': 1, 'how': 1, 'are': 1, 'fine': 1, 'morning': 1, 'night': 1, 'thank': 1, 'love': 1, 'machine': 1, 'what': 1, 'is

In [169]:
tgt_vocab=build_vocab([tgt for src, tgt in pairs])

src_vocab_size=len(src_vocab)
tgt_vocab_size=len(tgt_vocab)

print("Source vocabulary:",src_vocab_size)
print("Target vocabulary:",tgt_vocab_size)

Counter({'नमस्ते': 1})
Counter({'नमस्ते': 1, 'तपाईंलाई': 1, 'कस्तो': 1, 'छ': 1})
Counter({'छ': 2, 'नमस्ते': 1, 'तपाईंलाई': 1, 'कस्तो': 1, 'मलाई': 1, 'सन्चै': 1})
Counter({'छ': 2, 'नमस्ते': 1, 'तपाईंलाई': 1, 'कस्तो': 1, 'मलाई': 1, 'सन्चै': 1, 'शुभ': 1, 'प्रभात': 1})
Counter({'छ': 2, 'शुभ': 2, 'नमस्ते': 1, 'तपाईंलाई': 1, 'कस्तो': 1, 'मलाई': 1, 'सन्चै': 1, 'प्रभात': 1, 'रात्री': 1})
Counter({'छ': 2, 'शुभ': 2, 'नमस्ते': 1, 'तपाईंलाई': 1, 'कस्तो': 1, 'मलाई': 1, 'सन्चै': 1, 'प्रभात': 1, 'रात्री': 1, 'धन्यवाद': 1})
Counter({'छ': 2, 'शुभ': 2, 'नमस्ते': 1, 'तपाईंलाई': 1, 'कस्तो': 1, 'मलाई': 1, 'सन्चै': 1, 'प्रभात': 1, 'रात्री': 1, 'धन्यवाद': 1, 'म': 1, 'सिक्दै': 1, 'छु': 1})
Counter({'छ': 2, 'मलाई': 2, 'शुभ': 2, 'नमस्ते': 1, 'तपाईंलाई': 1, 'कस्तो': 1, 'सन्चै': 1, 'प्रभात': 1, 'रात्री': 1, 'धन्यवाद': 1, 'म': 1, 'सिक्दै': 1, 'छु': 1, 'मेसिन': 1, 'लर्निङ': 1, 'मन': 1, 'पर्छ': 1})
Counter({'छ': 2, 'मलाई': 2, 'शुभ': 2, 'नमस्ते': 1, 'तपाईंलाई': 1, 'कस्तो': 1, 'सन्चै': 1, 'प्रभात': 1, 'रात्री': 1, 'धन

In [170]:
src_vocab.items()

dict_items([('<PAD>', 0), ('<UNK>', 1), ('<BOS', 2), ('<EOS', 3), ('hello', 4), ('how', 5), ('are', 6), ('you', 7), ('i', 8), ('am', 9), ('fine', 10), ('good', 11), ('morning', 12), ('night', 13), ('thank', 14), ('learning', 15), ('love', 16), ('machine', 17), ('what', 18), ('is', 19), ('your', 20), ('name', 21), ('my', 22), ('john', 23), ('like', 24), ('python', 25), ('interesting', 26), ('old', 27), ('where', 28), ('going', 29), ('a', 30), ('student', 31), ('she', 32), ('happy', 33), ('he', 34), ('teacher', 35), ('want', 36), ('to', 37), ('learn', 38), ('see', 39), ('tomorrow', 40)])

In [171]:
src_dx_to_word={idx:word for word,idx in src_vocab.items()}
print(src_dx_to_word)

tgt_dx_to_word={idx:word for word,idx in tgt_vocab.items()}
print(tgt_dx_to_word)

{0: '<PAD>', 1: '<UNK>', 2: '<BOS', 3: '<EOS', 4: 'hello', 5: 'how', 6: 'are', 7: 'you', 8: 'i', 9: 'am', 10: 'fine', 11: 'good', 12: 'morning', 13: 'night', 14: 'thank', 15: 'learning', 16: 'love', 17: 'machine', 18: 'what', 19: 'is', 20: 'your', 21: 'name', 22: 'my', 23: 'john', 24: 'like', 25: 'python', 26: 'interesting', 27: 'old', 28: 'where', 29: 'going', 30: 'a', 31: 'student', 32: 'she', 33: 'happy', 34: 'he', 35: 'teacher', 36: 'want', 37: 'to', 38: 'learn', 39: 'see', 40: 'tomorrow'}
{0: '<PAD>', 1: '<UNK>', 2: '<BOS', 3: '<EOS', 4: 'नमस्ते', 5: 'तपाईंलाई', 6: 'कस्तो', 7: 'छ', 8: 'मलाई', 9: 'सन्चै', 10: 'शुभ', 11: 'प्रभात', 12: 'रात्री', 13: 'धन्यवाद', 14: 'म', 15: 'सिक्दै', 16: 'छु', 17: 'मेसिन', 18: 'लर्निङ', 19: 'मन', 20: 'पर्छ', 21: 'तपाईंको', 22: 'नाम', 23: 'के', 24: 'हो', 25: 'मेरो', 26: 'जोन', 27: 'पाइथन', 28: 'रोचक', 29: 'उमेर', 30: 'कति', 31: 'तपाईं', 32: 'कहाँ', 33: 'जाँदै', 34: 'हुनुहुन्छ', 35: 'विद्यार्थी', 36: 'हुँ', 37: 'उनी', 38: 'खुसी', 39: 'छिन्', 40: 'शिक्षक

In [172]:
PAD_IDX=0
UNK_IDX=1
BOS_IDX=2
EOS_IDX=3

In [173]:
def encode(sentence,vocab):
    tokens=tokenize(sentence)
    ids=[BOS_IDX]
    
    for word in tokens:
        ids.append(vocab.get(word,UNK_IDX))
    ids.append(EOS_IDX)
    return ids

In [174]:
print(encode("i love machine learning",src_vocab))

[2, 8, 16, 17, 15, 3]


In [175]:
class TranslationDatatset(Dataset):
    def __init__(self,pairs,src_vocab,tgt_vocab):
        self.paris=pairs
        self.src_vocab=src_vocab
        self.tgt_vocab=tgt_vocab
        
    def __len__(self):
        return len(self.paris)
    
    def __getitem__(self, idx):
        src_sentence,tgt_sentence=self.paris[idx]
        src_ids=encode(src_sentence,self.src_vocab)
        print(src_ids)
        
        tgt_ids=encode(tgt_sentence,self.tgt_vocab)
        print(tgt_ids)
        
        return (torch.tensor(src_ids),torch.tensor(tgt_ids))

In [176]:
def collect(batch):
    src_batch,tgt_batch=zip(*batch)
    src_batch=nn.utils.rnn.pad_sequence(src_batch,batch_first=True,padding_value=PAD_IDX)
    tgt_batch=nn.utils.rnn.pad_sequence(tgt_batch,batch_first=True,padding_value=PAD_IDX)
    
    return src_batch,tgt_batch

In [177]:
dataset=TranslationDatatset(pairs,src_vocab,tgt_vocab)
print(dataset)

In [178]:
loader=DataLoader(dataset,batch_size=4,shuffle=True,collate_fn=collect)

In [179]:
src,tgt=next(iter(loader))
print("Source:",src.shape)
print("Target:",tgt.shape)

[2, 8, 36, 37, 38, 3]
[2, 14, 42, 43, 3]
[2, 17, 15, 19, 26, 3]
[2, 17, 18, 28, 7, 3]
[2, 8, 24, 25, 3]
[2, 8, 27, 19, 20, 3]
[2, 8, 9, 30, 31, 3]
[2, 14, 35, 36, 3]
Source: torch.Size([4, 6])
Target: torch.Size([4, 6])


In [180]:
tgt_input=tgt[:,:-1]
tgt_output=tgt[:,1:]
print(tgt_input)

tensor([[ 2, 14, 42, 43,  3],
        [ 2, 17, 18, 28,  7],
        [ 2,  8, 27, 19, 20],
        [ 2, 14, 35, 36,  3]])


In [181]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()
        pe=torch.zeros(max_len,d_model)
        position=torch.arange(max_len).unsqueeze(1)
        div_term=torch.exp(torch.arange(0,d_model,2)* (-math.log(10000.0)/d_model))
        pe[:,0::2]=torch.sin(position * div_term)
        pe[:,1::2]=torch.cos(position* div_term)
        pe=pe.unsqueeze(0)
        self.register_buffer("pe",pe)
    
    def forward(self,x):
        return x+self.pe[:,:x.size(1)]

In [182]:
class ScaledDotProductAttention(nn.Module):
    def __init__(self, dropout=0.1):
        super().__init__()
        self.dropout=nn.Dropout(dropout)
    
    def forward(self,Q,K,V,mask=None):
        d_k=Q.size(-1)
        scores=torch.matmul(Q,K.transpose(-2,-1))/math.sqrt(d_k)
        if mask is not None:
            scores=scores.masked_fill(~mask,torch.finfo(scores.dtype).min)
        weights=torch.softmax(scores,dim=-1)
        weights=self.dropout(weights)
        output=torch.matmul(weights,V)
        return output,weights

In [183]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model,num_heads,dropout=0.1):
        super().__init__()
        assert d_model% num_heads==0
        self.d_model=d_model
        self.num_heads=num_heads
        self.d_k=d_model// num_heads
        self.Wq=nn.Linear(d_model,d_model)
        self.Wk=nn.Linear(d_model,d_model)
        self.Wv=nn.Linear(d_model,d_model)
        self.Wo=nn.Linear(d_model,d_model)
        self.attention=ScaledDotProductAttention(dropout)
        
    def split_heads(self,x):
        B,S,D=x.shape
        x=x.reshape(B,S,self.num_heads,self.d_k)
        
        return x.transpose(1,2)
    
    def forward(self,query,key,value,mask=None):
        Q=self.split_heads(self.Wq(query))
        K=self.split_heads(self.Wk(key))
        V=self.split_heads(self.Wv(value))
        
        output,weightts=self.attention(Q,K,V,mask)
        output=output.transpose(1,2).contiguous()
        B,S,H,D,=output.shape
        output=output.reshape(B,S,H*D)
        output=self.Wo(output)
        return output,weightts

In [184]:
class FeedForwardNetwork(nn.Module):
    def __init__(self,d_model,d_ff,dropout=0.1):
        super().__init__()
        self.fc1=nn.Linear(d_model,d_ff)
        self.fc2=nn.Linear(d_ff,d_model)
        self.dropout=nn.Dropout(dropout)
        
    def forward(self,x):
        x=self.fc1(x)
        x=F.relu(x)
        x=self.dropout(x)
        x=self.fc2(x)
        return x

In [185]:
class EncodedLayer(nn.Module):
    def __init__(self, d_model,num_heads,d_ff,dropout=0.1):
        super().__init__()
        self.attention=MultiHeadAttention(d_model,num_heads,dropout)
        self.norm1=nn.LayerNorm(d_model)
        self.ffn=FeedForwardNetwork(d_model,d_ff,dropout)
        self.norm2=nn.LayerNorm(d_model)
        self.dropout=nn.Dropout(dropout)
        
    def forward(self,x,src_mask):
        attn_output=self.attention(x,x,x,src_mask)
        x=self.norm1(x+self.dropout(attn_output))
        ffn_output=self.ffn(x)
        x=self.norm2(x+self.dropout(ffn_output))
        return x

In [186]:
class Encoder(nn.Module):
    def __init__(self, vocab_size,d_model,num_heads,d_ff,num_layers,dropout=0.1):
        super().__init__()
        self.embedding=nn.Embedding(vocab_size,d_model,padding_idx=PAD_IDX)
        self.positional=PositionalEncoding(d_model)
        self.layers=nn.ModuleList([EncodedLayer(d_model,num_heads,d_ff,dropout) for _ in range(num_layers)])
        self.dropout=nn.Dropout(dropout)
        self.scale=math.sqrt(d_model)
    
    def forward(self,src,src_mask):
        x=self.embedding(src)*self.scale
        x=self.positional(x)
        x=self.dropout(x)
        
        for layer in self.layers:
            x=layer(x,src_mask)
        return x

In [187]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model,num_heads,d_ff,dropout=0.1):
        super().__init__()
        self.self_attention=MultiHeadAttention(d_model,num_heads,dropout)
        self.norm1=nn.LayerNorm(d_model)
        self.cross_attention=MultiHeadAttention(d_model,num_heads,dropout)
        
        self.num1=nn.LayerNorm(d_model)
        self.cross_attention=MultiHeadAttention(d_model,num_heads,dropout)
        
        self.norm2=nn.LayerNorm(d_model)
        self.ffn=FeedForwardNetwork(d_model,d_ff,dropout)
        self.norm3=nn.LayerNorm(d_model)
        self.dropout=nn.Dropout(dropout)
        
    def forward(self,x,encoder_output,tgt_mask,src_mask):
        # masked self attention
        attn_output=self.self_attention(x,x,x,tgt_mask,src_mask)
        x=self.norm1(x+self.dropout(attn_output))
        
        #cross attention
        cross_output,_=self.cross_attention(x,encoder_output,encoder_output,src_mask)
        x=self.norm2(x+self.dropout(cross_output))
        
        #feed forward
        ffn_output=self.ffn(x)
        x=self.norm3(x+self.dropout(ffn_output))
        return x        

In [188]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model,num_heads,d_ff,num_layers,dropout=0.1):
        super().__init__()
        self.embedding=nn.Embedding(vocab_size,d_model,padding_idx=PAD_IDX)
        self.positional=PositionalEncoding(d_model)
        self.layers=nn.ModuleList([DecoderLayer(d_model,num_heads,d_ff,dropout) for _ in range(num_layers)])
        
        self.dropout=nn.Dropout(dropout)
        self.scale=math.sqrt(d_model)
        
        
    def forward(self,tgt,encoder_output,tgt_mask,src_mask):
        x=self.embedding(tgt)* self.scale
        x=self.positional(x)
        x=self.dropout(x)
        
        for layer in self.layers:
            x=layer(x,encoder_output,tgt_mask,src_mask)
        return x

In [189]:
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size,d_model=128,num_heads=4,d_ff=512,num_layers=2,dropout=0.1):
        super().__init__()
        self.encoder=Encoder(src_vocab_size,d_model,num_heads,d_ff,num_layers,dropout)
        self.decoder=Decoder(tgt_vocab_size,d_model,num_heads,d_ff,num_layers,dropout)
        self.output_layer=nn.Linear(d_model,tgt_vocab_size)
        
    def forward(self,src,tgt):
        src_mask=(src!=PAD_IDX)[:,None,None,:]
        
        #target padding mask
        tgt_padding_mask=(tgt!=PAD_IDX)[:None,None,:]
        
        #casual mask
        T=tgt.size(1)
        
        casual_mask=torch.tril(torch.ones(T,T,device=tgt.device,dtype=torch.bool))[None,None,:]
        
        tgt_mask=(tgt_padding_mask & casual_mask)
        
        #encoder
        encoder_output=self.encoder(src,src_mask)
        
        #decoder
        decoder_output=self.decoder(tgt,encoder_output,tgt_mask,src_mask)
        
        #vocab prediction
        logits=self.output_layer(decoder_output)
        return logits      

In [190]:
model=Transformer(
    src_vocab_size=100,
    tgt_vocab_size=100,
    d_model=128,
    num_heads=4,
    d_ff=512,
    num_layers=2,
    dropout=0.1
)
print(model)

Transformer(
  (encoder): Encoder(
    (embedding): Embedding(100, 128, padding_idx=0)
    (positional): PositionalEncoding()
    (layers): ModuleList(
      (0-1): 2 x EncodedLayer(
        (attention): MultiHeadAttention(
          (Wq): Linear(in_features=128, out_features=128, bias=True)
          (Wk): Linear(in_features=128, out_features=128, bias=True)
          (Wv): Linear(in_features=128, out_features=128, bias=True)
          (Wo): Linear(in_features=128, out_features=128, bias=True)
          (attention): ScaledDotProductAttention(
            (dropout): Dropout(p=0.1, inplace=False)
          )
        )
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
        (ffn): FeedForwardNetwork(
          (fc1): Linear(in_features=128, out_features=512, bias=True)
          (fc2): Linear(in_features=512, out_features=128, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (norm2): LayerNorm((128,), eps=1e-05, elementwise

In [191]:
criterion=nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer=torch.optim.Adam(model.parameters(),lr=0.0003,betas=(0.9,0.98),eps=1e-9)

In [192]:
for src,tgt in loader:
    tgt_input=tgt[:,:-1]
    print(tgt_input)

[2, 22, 21, 19, 23, 3]
[2, 25, 22, 26, 24, 3]
[2, 18, 19, 20, 21, 3]
[2, 21, 22, 23, 24, 3]
[2, 8, 9, 10, 3]
[2, 8, 9, 7, 3]
[2, 28, 6, 7, 29, 3]
[2, 31, 32, 33, 34, 3]
tensor([[ 2, 25, 22, 26, 24],
        [ 2, 21, 22, 23, 24],
        [ 2,  8,  9,  7,  3],
        [ 2, 31, 32, 33, 34]])
[2, 8, 16, 17, 15, 3]
[2, 8, 17, 18, 19, 20, 3]
[2, 5, 27, 6, 7, 3]
[2, 21, 29, 30, 24, 3]
[2, 8, 9, 30, 31, 3]
[2, 14, 35, 36, 3]
[2, 14, 7, 3]
[2, 13, 3]
tensor([[ 2,  8, 17, 18, 19, 20],
        [ 2, 21, 29, 30, 24,  3],
        [ 2, 14, 35, 36,  3,  0],
        [ 2, 13,  3,  0,  0,  0]])
[2, 39, 7, 40, 3]
[2, 44, 45, 3]
[2, 17, 15, 19, 26, 3]
[2, 17, 18, 28, 7, 3]
[2, 34, 19, 30, 35, 3]
[2, 37, 40, 41, 3]
[2, 8, 24, 25, 3]
[2, 8, 27, 19, 20, 3]
tensor([[ 2, 44, 45,  3,  0],
        [ 2, 17, 18, 28,  7],
        [ 2, 37, 40, 41,  3],
        [ 2,  8, 27, 19, 20]])
[2, 11, 13, 3]
[2, 10, 12, 3]
[2, 32, 19, 33, 3]
[2, 37, 38, 39, 3]
[2, 5, 6, 7, 3]
[2, 5, 6, 7, 3]
[2, 8, 9, 15, 3]
[2, 14, 15, 16, 3]


In [193]:
epochs=300

for epoch in range(epochs):
    model.train()
    total_loss=0
    
    for src,tgt in loader:
        src=src.to(device)
        tgt=tgt.to(device)
        
        tgt_input=tgt[:,:-1]
        tgt_output=tgt[:,1:]
        
        optimizer.zero_grad()
        
        logits=model(src,tgt_input)
        loss=criterion(logits.reshape(-1,tgt_vocab_size),tgt_output.reshape(-1))
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(),max_norm=1.0)
        optimizer.step()
        
        total_loss+=loss.item()
        
    avg_loss=total_loss/len(loader)
    if (epoch +1)%25==0:
        print(
            f"Epoch {epoch+1}/{epochs},"
            f"Loss:{ avg_loss:.4f}"
        )
    

[2, 28, 6, 7, 29, 3]
[2, 31, 32, 33, 34, 3]
[2, 18, 19, 20, 21, 3]
[2, 21, 22, 23, 24, 3]
[2, 5, 6, 7, 3]
[2, 5, 6, 7, 3]
[2, 32, 19, 33, 3]
[2, 37, 38, 39, 3]


TypeError: dropout(): argument 'input' (position 1) must be Tensor, not tuple

In [ ]:
def generate_text(model_sentence,src,vocab,tgt_idx_to_word,max_len=30):
    model.eval()
    src_ids=encode(sentence,src_vocab)
    src=torch.tensor([src_ids],dtype=torch.long).to(device)
    
    src_mask=(src!=PAD_IDX)[:,None,None,:]
    with torch.no_grad():
        encoder_output=model.encoder(src,src_mask)
        generated=torch.tensor([[BOS_IDX]],dtype=torch.long,device=device)
        
        for step in range(max_len):
            T=generated.size(1)
            
            casual_mask=torch.tril(torch.ones(T,T,dtype=torch.bool,device=device))[[None,None,:,:]]
            
            #target passing mask
            tgt_padding_mask=(generated !=PAD_IDX)[:,None,None,:]
            tgt_mask=(casual_mask & tgt_padding_mask)
            
            #Decoder forward pass
            decoder_output=model.decoder(generated,encoder_output,)
            logits=model.output_layer(decoder_output)
            
            next_token_logits=logits[:,-1,:]
            next_token_logits[:,PAD_IDX]=float("-inf")
            next_token_logits[:,BOS_IDX]=float("-inf")
            
            next_token=torch.argmax(next_token_logits,dim=1)
            generated=torch.cat([generated,next_token.unsqueeze(1)],dim=1)
            predicted_id=next_token.item()
            
            predicted_word=tgt_idx_to_word.get(predicted_id,"<UNK")
            print(f"Step {step+1}: {predicted_word}")
            
            #Stop if EOS is generated
            if predicted_id==EOS_IDX:
                break
            
    #step 6 convert generated IDS into words
    generated_ids=generated[0].tolist()
    words=[]
    
    for token_id in generated_ids:
        if token_id==BOS_IDX:
            continue
        
        if token_id==EOS_IDX:
            break
        
        if token_id==PAD_IDX:
            continue
        
        words.append(tgt_idx_to_word.get(token_id,"<UNK>"))
    return " ".join(words)   
        

In [ ]:
sentence="he is a teacher"

translation=generate_text(model,sentence,src_vocab,tgt_dx_to_word)
print("\nInput:",sentence)
print("Generated translation:", translation)

VIT Transformer